# 🚀 DervGen Cloud Workspace Bootstrap

Welcome to the **DervGen Unified Cloud Workspace** on Google Colab.
This notebook initializes your cloud development runtime with full toolchains for **C++20, CUDA, PyTorch, Bazel (9.3.1), and Go**.

### ✨ Quick Features
- ⚡ **GPU Environment Verification**: Auto-detects NVIDIA GPUs (T4 / V100 / A100 / H100) and CUDA capabilities.
- 💾 **Persistence via Google Drive**: Optional mount to keep datasets and persistent models.
- 🛠️ **Automated Toolchain Setup**: One-click install for Bazel, Clang, Go, and Python libraries.
- 📦 **DervGen Ecosystem Integration**: Selective cloning of `aether`, `fulgur`, `caelum-core`, `vg-core`, etc.
- 💻 **Remote IDE Support**: Start a VS Code Tunnel to code from your local desktop VS Code or Antigravity.

## 1. ⚙️ Configuration & Hardware Verification
Verify GPU availability and configure runtime options.

In [ ]:
#@title Workspace Launch Parameters { run: "auto" }
MOUNT_GOOGLE_DRIVE = True #@param {type:"boolean"}
SETUP_TOOLCHAINS = True #@param {type:"boolean"}
INSTALL_BLENDER = True #@param {type:"boolean"}
START_VSCODE_TUNNEL = False #@param {type:"boolean"}
START_TAILSCALE = False #@param {type:"boolean"}
TAILSCALE_HOSTNAME = "dervgen-colab-gpu" #@param {type:"string"}
STAGE_EVALUATION_DATASETS = True #@param {type:"boolean"}
INSTALL_ML_TOOLS = True #@param {type:"boolean"}
ML_TOOLS_PRESET = "full" #@param ["full", "llm", "cv", "tabular", "eval_only", "minimal"]
CLONE_REPOSITORIES = "aether,fulgur" #@param {type:"string"}

import os
import sys
import subprocess

# Check NVIDIA GPU
try:
    gpu_info = subprocess.check_output(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"]).decode("utf-8").strip()
    print(f"✅ NVIDIA GPU Detected: {gpu_info}")
except Exception:
    print("⚠️ No NVIDIA GPU detected. Ensure Runtime -> Change runtime type -> Hardware accelerator is set to GPU.")

# Check PyTorch CUDA
try:
    import torch
    print(f"✅ PyTorch Version: {torch.__version__} | CUDA Available: {torch.cuda.is_available()}")
    if torch.cuda.is_available():
        print(f"   Device Name: {torch.cuda.get_device_name(0)}")
except ImportError:
    print("PyTorch not loaded.")

## 2. 💾 Mount Google Drive (Optional Persistence)

In [ ]:
if MOUNT_GOOGLE_DRIVE:
    from google.colab import drive
    drive_path = "/content/drive"
    if not os.path.exists(drive_path):
        drive.mount(drive_path)
        print("✅ Google Drive successfully mounted at /content/drive")
    else:
        print("Google Drive already mounted.")

## 3. 📥 Clone & Setup Colab Workenv Tools

In [ ]:
COLAB_DIR = "/content/colab-workenv"

# Configure Git Credentials from Colab Secrets if available
try:
    from google.colab import userdata
    try:
        github_token = userdata.get('GITHUB_TOKEN')
        os.environ['GITHUB_TOKEN'] = github_token
        print("✅ Loaded GITHUB_TOKEN from Colab Secrets.")
    except Exception:
        print("ℹ️ GITHUB_TOKEN not found in Secrets. Using public HTTPS.")
except ImportError:
    pass

if not os.path.exists(COLAB_DIR):
    print(f"Cloning sautille-stroke/colab-workenv into {COLAB_DIR}...")
    token = os.environ.get('GITHUB_TOKEN', '')
    repo_url = f"https://{token}@github.com/sautille-stroke/colab-workenv.git" if token else "https://github.com/sautille-stroke/colab-workenv.git"
    !git clone {repo_url} {COLAB_DIR}
else:
    print(f"colab-workenv already present at {COLAB_DIR}. Updating...")
    !cd {COLAB_DIR} && git pull

%cd {COLAB_DIR}
os.environ["WORKSPACE_ROOT"] = COLAB_DIR
os.environ["COLAB_ROOT"] = COLAB_DIR


## 4. 🛠️ Install Toolchains (Bazelisk, LLVM/Clang 18, Go 1.23+, Blender 4.3 LTS, Dependencies)

Installs and validates the unified DervGen ecosystem toolchain:
- **Bazel**: Bazelisk (9.3.1 / Bzlmod) at `/usr/local/bin/bazel`
- **LLVM/Clang 18**: C++20 concepts, lld linker, libc++, libomp runtime
- **Go 1.23+**: Official modern Go toolchain for narrative-rail and backend services
- **Blender 4.3.2 LTS**: Headless Cycles path tracer reference oracle
- **Diagnostics**: Runs deep health check (`check_env.sh`) verifying all compilers & runtimes


In [ ]:
if SETUP_TOOLCHAINS:
    flags = ""
    if not INSTALL_BLENDER:
        flags += " --skip-blender"
    !bash {COLAB_DIR}/setup_env.sh {flags}
    !bash {COLAB_DIR}/scripts/check_env.sh


## 5. 📦 Clone Ecosystem Repositories (e.g. Aether, Fulgur)

In [ ]:
if CLONE_REPOSITORIES.strip():
    target_dir = "/content/dervgen"
    !bash {COLAB_DIR}/repos/clone_repos.sh "{target_dir}" "{CLONE_REPOSITORIES}" "true"


## 6. 📊 Stage Evaluation Datasets to Google Drive (/dervgen/data/...)
Stage the standard evaluation benchmark suite: RFC 0053 SWE-bench for 3D (90 cases), production closed-loop cases (Mech Ranger, Cyber Hunter, Cinematic Orbit), and 3D scenes directly into Google Drive cloud storage (Zero local disk consumption).

In [ ]:
if MOUNT_GOOGLE_DRIVE and STAGE_EVALUATION_DATASETS:
    print("🚀 Staging evaluation datasets & benchmark suites directly into Google Drive...")
    !python3 {COLAB_DIR}/init_datasets.py
    
    index_md = "/content/drive/MyDrive/dervgen/data/INDEX.md"
    if os.path.exists(index_md):
        with open(index_md, "r", encoding="utf-8") as f:
            print("\n--- [Live Dataset & Evaluation Index] ---")
            print(f.read())


## 7. 🛠️ Pre-fetch ML Training & Evaluation Toolchain to Google Drive
Downloads **wheel packages** for training frameworks (Transformers, Lightning, PEFT, TRL, bitsandbytes), evaluation metrics (TorchMetrics, Evaluate, ROUGE, BERTScore), experiment trackers (W&B, TensorBoard, MLflow), classical ML (LightGBM, XGBoost, CatBoost), tokenizers (SentencePiece, tiktoken), and HPO (Optuna) directly into Google Drive (`/dervgen/tools/wheels/`).

Preset options: `full` (all), `llm` (LLM fine-tuning), `cv` (vision), `tabular` (tree models), `eval_only` (metrics only), `minimal` (bare essentials).

> **Note**: This step only downloads wheels to Drive for caching. To install from cached wheels later, run: `!python3 {COLAB_DIR}/init_tools.py --install`

In [ ]:
if INSTALL_ML_TOOLS and MOUNT_GOOGLE_DRIVE:
    import importlib, sys
    sys.path.insert(0, COLAB_DIR)
    import init_tools
    importlib.reload(init_tools)
    
    print(f"📦 Pre-fetching ML toolchain wheels to Google Drive (preset: {ML_TOOLS_PRESET})...")
    init_tools.run_download(preset=ML_TOOLS_PRESET)
elif INSTALL_ML_TOOLS and not MOUNT_GOOGLE_DRIVE:
    print("⚠️ Google Drive not mounted. ML tools download requires Drive. Skipping.")
else:
    print("ℹ️ ML tools pre-fetch skipped. Set INSTALL_ML_TOOLS=True to enable.")


In [ ]:
# 💡 Quick-reference: Install from cached wheels & GPU memory helpers
import torch, gc

def install_from_cache(categories=None):
    """Install ML tools from cached wheels on Google Drive (offline, no network)."""
    import init_tools
    init_tools.run_install_from_cache(categories=categories)

def print_gpu_memory():
    """Print current GPU memory usage."""
    if torch.cuda.is_available():
        allocated = torch.cuda.memory_allocated() / (1024 ** 3)
        reserved = torch.cuda.memory_reserved() / (1024 ** 3)
        total = torch.cuda.get_device_properties(0).total_memory / (1024 ** 3)
        print(f"GPU Memory: {allocated:.2f} GB alloc | {reserved:.2f} GB reserved | {total:.2f} GB total")
    else:
        print("No GPU detected")

def clear_gpu_cache():
    """Free unused GPU memory and Python garbage."""
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.synchronize()
    print("GPU cache cleared")
    print_gpu_memory()

def estimate_model_memory(param_count_billions: float, precision: str = 'fp16') -> str:
    """Estimate VRAM needed for a model."""
    bytes_per_param = {"fp32": 4, "fp16": 2, "bf16": 2, "int8": 1, "int4": 0.5}
    bpp = bytes_per_param.get(precision, 2)
    vram_gb = param_count_billions * bpp
    training_vram_gb = vram_gb * 4
    return f"Inference: ~{vram_gb:.1f} GB | Full Train: ~{training_vram_gb:.1f} GB | LoRA: ~{vram_gb * 1.2:.1f} GB"

print("Helpers ready:")
print("  install_from_cache()                     - Install ML tools from cached wheels on Drive")
print('  install_from_cache(categories=["A","B"])  - Install specific categories only')
print("  print_gpu_memory()                       - Show current VRAM usage")
print("  clear_gpu_cache()                        - Free unused GPU memory")
print('  estimate_model_memory(7.0, "int4")       - Estimate VRAM for 7B model in 4-bit')
print()
print_gpu_memory()

## 8. 💻 Launch VS Code Desktop Tunnel (Optional)
Run this cell to start a Microsoft VS Code Tunnel so you can edit code directly in your local VS Code / Antigravity editor!

In [ ]:
if START_VSCODE_TUNNEL:
    !bash {COLAB_DIR}/vscode_tunnel.sh "dervgen-colab-gpu"

## 9. 🔒 Connect via Tailscale SSH (Optional)
Start Tailscale and Tailscale SSH to connect directly from your local terminal or SSH client via P2P WireGuard.

In [ ]:
if START_TAILSCALE:
    ts_authkey = os.environ.get("TAILSCALE_AUTHKEY", "")
    try:
        from google.colab import userdata
        ts_authkey = userdata.get("TAILSCALE_AUTHKEY") or ts_authkey
    except Exception:
        pass
    !bash {COLAB_DIR}/tailscale.sh "{ts_authkey}" "{TAILSCALE_HOSTNAME}"


## 10. 🔄 Git & Drive Synchronization Helpers
Use these helper functions to push your work to GitHub or backup to Google Drive.

In [ ]:
def sync_to_github(commit_message: str = "wip: update from colab session"):
    !bash {COLAB_DIR}/git_sync.sh "{commit_message}"

def backup_to_drive():
    !python3 {COLAB_DIR}/drive_sync.py

print("✅ Sync helpers ready! Example usage:")
print("   sync_to_github('feat: test new shader algorithm')")
print("   backup_to_drive()")